# Repository Recommender - Data Exploration & Baseline Model

This notebook demonstrates the DS workflow for the git-query recommender:
1. Load repository dataset from the gateway API
2. Explore data distributions
3. Extract features for ranking
4. Train a baseline ranking model
5. Evaluate with IR metrics

In [ ]:
import sys
sys.path.insert(0, '../../..')  # Add project root to path

import os
from dotenv import load_dotenv
load_dotenv(dotenv_path='../../../.env')  # Load .env from project root

from src.recommender.data import RepoDataset, FeatureExtractor
import pandas as pd
import numpy as np

In [ ]:
# Fetch from gateway (first time)
# ds = RepoDataset.from_gateway(
#     url=os.getenv("API_BASE_URL"),
#     api_key=os.getenv("GATEWAY_API_KEY"),
#     max_repos=1000
# )
# ds.save("data/repos_1000.parquet")

# Load from local cache (faster)
ds = RepoDataset.from_local("data/repos_1000.parquet")
print(f"Loaded {len(ds)} repositories")
ds.summary()

In [ ]:
df = ds.to_dataframe()
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nStar distribution:")
print(df["stars"].describe())

In [ ]:
df["language"].value_counts().head(20).plot(kind="barh", figsize=(10, 6), title="Top 20 Languages")

In [ ]:
fe = FeatureExtractor()
features = fe.extract_all(df, query="python web framework")
print(f"Features shape: {features.shape}")
print(f"Features: {list(features.columns)}")
features.head()

In [ ]:
# Check which features correlate with stars (proxy for quality)
correlations = features.corrwith(df["stars"]).sort_values(ascending=False)
print("Feature correlations with stars:")
print(correlations)

## Baseline Ranking Model

Train a simple LightGBM ranker using topic-based synthetic relevance labels.

For each repo, relevance = topic overlap with a test query. This gives us ground truth for training without user data.

In [ ]:
# Create synthetic relevance labels based on topic overlap
# In production, these would come from user clicks/saves
test_queries = [
    "python machine learning",
    "javascript web framework", 
    "go microservices",
    "rust systems programming",
    "typescript react frontend",
]

# For each query, score repos by topic overlap
query = test_queries[0]
fe_query = FeatureExtractor()
relevance = fe_query.topic_overlap(df, query.split())
print(f"Query: '{query}'")
print(f"Repos with >0 overlap: {(relevance > 0).sum()}")

In [ ]:
from sklearn.model_selection import train_test_split

# Use features + synthetic relevance
X = features.fillna(0)
y = relevance.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Simple baseline: score = weighted combination of features
# DS would replace this with LightGBM/XGBoost later

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import ndcg_score

model = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Predict
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"Train NDCG: {ndcg_score([y_train.values], [y_pred_train]):.4f}")
print(f"Test NDCG:  {ndcg_score([y_test.values], [y_pred_test]):.4f}")

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importance:")
print(importance)
importance.head(10).plot(kind="barh", figsize=(10, 4), title="Top 10 Feature Importances")

## Next Steps

- [ ] Try LightGBM ranker with `lambdarank` objective
- [ ] Add more features (readme analysis, contributor count, issue stats)
- [ ] Test with more queries and build a proper evaluation set
- [ ] Compare pretrained cross-encoder vs custom model
- [ ] Export trained model for production serving